## 1. Initialize Project Environment
Import dependencies and configure paths for FASTQ input.

In [9]:
from __future__ import annotations

import gzip
import logging
from pathlib import Path
from typing import Dict

import pandas as pd
from Bio import SeqIO

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
try:
    import Bio

    print("biopython", Bio.__version__)
except Exception as exc:
    logging.error("Biopython import failed: %s", exc)

pandas 2.2.3
biopython 1.85


## 2. Define Configuration Parameters
Set paths to FASTQ input and output report.

In [10]:
from dataclasses import dataclass, asdict
from typing import Optional


def locate_repo_root() -> Path:
    """Find the repository root by looking for data/work directory."""
    here = Path().resolve()
    for base in [here, *here.parents]:
        if (base / "data/work").exists():
            return base
    raise FileNotFoundError("Could not locate repository root")


@dataclass
class FastqConfig:
    handle: str
    fastq_path: Optional[Path] = None
    export_dir: Path = Path("artifacts")

    def __post_init__(self):
        if self.fastq_path is None:
            repo_root = locate_repo_root()
            self.fastq_path = (
                repo_root / f"data/work/{self.handle}/lab03/your_reads.fastq.gz"
            )

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["fastq_path"] = str(info["fastq_path"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = FastqConfig(handle="AndreiCod")
CONFIG.describe()

{'handle': 'AndreiCod',
 'fastq_path': '/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab03/your_reads.fastq.gz',
 'export_dir': 'artifacts'}

In [11]:
# Verify FASTQ file exists
if CONFIG.fastq_path.exists():
    size_mb = CONFIG.fastq_path.stat().st_size / 1e6
    print(f"Found FASTQ: {CONFIG.fastq_path}")
    print(f"Size: {size_mb:.2f} MB")
else:
    logging.error(f"FASTQ not found: {CONFIG.fastq_path}")

Found FASTQ: /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab03/your_reads.fastq.gz
Size: 31.13 MB


## 3. Implement Core Functionality
Build utilities for computing FASTQ quality statistics.

In [12]:
def compute_fastq_stats(fastq_path: Path, is_gzipped: bool = True) -> Dict:
    """Compute QC statistics for a FASTQ file."""
    num_reads = 0
    total_length = 0
    total_n = 0
    total_phred = 0
    total_bases = 0

    # Open file appropriately
    if is_gzipped:
        reader = SeqIO.parse(gzip.open(fastq_path, "rt"), "fastq")
    else:
        reader = SeqIO.parse(str(fastq_path), "fastq")

    for record in reader:
        num_reads += 1
        seq_str = str(record.seq)
        total_length += len(seq_str)
        total_n += seq_str.count("N")
        phred = record.letter_annotations["phred_quality"]
        total_phred += sum(phred)
        total_bases += len(phred)

    # Compute final values (avoid division by zero)
    len_mean = total_length / num_reads if num_reads > 0 else 0.0
    n_rate = total_n / total_length if total_length > 0 else 0.0
    phred_mean = total_phred / total_bases if total_bases > 0 else 0.0

    return {
        "num_reads": num_reads,
        "mean_length": len_mean,
        "n_rate": n_rate,
        "mean_phred": phred_mean,
        "total_bases": total_bases,
        "total_n": total_n,
    }


print("QC function defined. Ready to compute statistics.")

QC function defined. Ready to compute statistics.


In [ ]:
# Run QC computation
is_gzipped = str(CONFIG.fastq_path).endswith(".gz")
print(f"Computing QC for: {CONFIG.fastq_path.name}")
print(f"Gzipped: {is_gzipped}")
print("This may take a few minutes for large files...")

stats = compute_fastq_stats(CONFIG.fastq_path, is_gzipped=is_gzipped)

# Display results
print("\n" + "=" * 40)
print("QC Results:")
print("=" * 40)
print(f"  Reads:       {stats['num_reads']:,}")
print(f"  Mean length: {stats['mean_length']:.2f} bp")
print(f"  N rate:      {stats['n_rate']:.4f}")
print(f"  Mean Phred:  {stats['mean_phred']:.2f}")

Computing QC for: your_reads.fastq.gz
Gzipped: True
This may take a few minutes for large files...


In [ ]:
# Display as a nice table
results_df = pd.DataFrame(
    [
        {"Metric": "Reads", "Value": f"{stats['num_reads']:,}"},
        {"Metric": "Mean length", "Value": f"{stats['mean_length']:.2f} bp"},
        {"Metric": "N rate", "Value": f"{stats['n_rate']:.4f}"},
        {"Metric": "Mean Phred", "Value": f"{stats['mean_phred']:.2f}"},
    ]
)
results_df

,Metric,Value
0,Reads,"1,170,794"
1,Mean length,36.00 bp
2,N rate,0.0002
3,Mean Phred,38.54


## 4. Validate with Unit Tests
Verify the QC computation produces expected results.

In [ ]:
def test_stats_values():
    """Verify computed stats are within reasonable bounds."""
    assert stats["num_reads"] > 0, "Should have at least one read"
    assert stats["mean_length"] > 0, "Mean length should be positive"
    assert 0 <= stats["n_rate"] <= 1, "N rate should be between 0 and 1"
    assert 0 <= stats["mean_phred"] <= 60, "Phred score should be realistic"


def test_expected_results():
    """Verify results match expected values for ERR000001."""
    assert stats["num_reads"] == 1170794, (
        f"Expected 1,170,794 reads, got {stats['num_reads']}"
    )
    assert abs(stats["mean_length"] - 36.0) < 0.1, "Expected mean length ~36 bp"
    assert abs(stats["mean_phred"] - 38.54) < 0.1, "Expected mean Phred ~38.54"


test_stats_values()
test_expected_results()
print("All inline tests passed.")

All inline tests passed.


## 5. Export Results
Save the QC report to a text file.

In [ ]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

out_file = EXPORT_DIR / "task3_fastq_qc_report.txt"

with open(out_file, "w", encoding="utf-8") as out:
    out.write(f"Reads: {stats['num_reads']}\n")
    out.write(f"Mean length: {stats['mean_length']:.2f}\n")
    out.write(f"N rate: {stats['n_rate']:.4f}\n")
    out.write(f"Mean Phred: {stats['mean_phred']:.2f}\n")

print(f"[OK] QC report saved to: {out_file.resolve()}")

[OK] QC report saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/03_formats&NGS/assignments/artifacts/task3_fastq_qc_report.txt
